<a href="https://colab.research.google.com/github/solsthecoder/NLP-LAB/blob/main/LAB02_NLP_FA23_BDS_045.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NLP LAB 02
# YASRAB TALIB
# FA23-BDS-045

In [ ]:
!pip install pdfplumber python-docx pandas requests nltk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 94.5 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0


In [ ]:
import re
import csv
import sys
import math
import pprint
import requests
import pdfplumber
import pandas as pd
from collections import Counter
from docx import Document
from docx.text.paragraph import Paragraph
from docx.oxml.ns import qn


In [ ]:
PDF_FILE = "sample.pdf"
DOCX_FILE = "sample.docx"
DOCX_A = "doc_a.docx"
DOCX_B = "doc_b.docx"

In [ ]:

# Common stopwords used in Tasks 1 and 6 (more than the required 20)
STOPWORDS = {
    "the", "a", "an", "and", "or", "but", "is", "are", "was", "were", "be", "been",
    "of", "in", "on", "at", "to", "for", "with", "by", "from", "as", "that", "this",
    "it", "its", "he", "she", "they", "we", "you", "i", "not", "have", "has", "had",
    "will", "would", "can", "which", "their", "there", "than", "then", "so", "if",
}

def read_pdf_pages(path):
    """Return list of strings, one string per PDF page."""
    with pdfplumber.open(path) as pdf:
        return [(page.extract_text() or "") for page in pdf.pages]



In [ ]:

# =====================================================================
# TASK 1: Multi-Page PDF Extraction with Word Frequency Analysis
# =====================================================================
print("=" * 60, "\nTASK 1\n" + "=" * 60)

# (a) extract text page by page
pages = read_pdf_pages(PDF_FILE)
print("(a) Total pages extracted:", len(pages))

# (b) merge, clean with re.sub, lowercase, split on whitespace
full_text = " ".join(pages)
cleaned = re.sub(r"[^a-zA-Z\s]", "", full_text)   # removes punctuation AND digits
tokens = cleaned.lower().split()
print("(b) Total tokens:", len(tokens))

# (c) frequency dictionary without stopwords
freq = {}
for word in tokens:
    if word not in STOPWORDS:
        freq[word] = freq.get(word, 0) + 1
top10 = sorted(freq.items(), key=lambda x: x[1], reverse=True)[:10]
print("(c) Top 10 words:")
for word, count in top10:
    print(f"    {word:<15} {count}")

# (d) keyword search, sentence by sentence, with page number
keyword = input("(d) Enter a keyword to search: ").strip().lower()
found = False
for page_no, page_text in enumerate(pages, start=1):
    sentences = re.split(r"(?<=[.!?])\s+", page_text)
    for sentence in sentences:
        if keyword and keyword in sentence.lower():
            print(f"    [Page {page_no}] {sentence.strip().replace(chr(10), ' ')}")
            found = True
if not found:
    print(f"    No sentence containing '{keyword}' was found.")



TASK 1
(a) Total pages extracted: 6
(b) Total tokens: 573
(c) Top 10 words:
    text            15
    language        10
    nlp             7
    words           7
    chapter         6
    word            5
    often           4
    called          4
    tokenization    4
    such            4
(d) Enter a keyword to search: word
    [Page 2] A token can be a word, a number or a punctuation mark.
    [Page 2] Advanced tokenizers handle contractions such as don't and it's, hyphenated words and abbreviations like Dr.
    [Page 4] Converting text to lowercase makes words like Language and language count as the same word.
    [Page 4] Stopwords are very common words such as the, is, and, of and to.
    [Page 4] However, for some tasks like translation, stopwords are important.
    [Page 4] The result is a tidy list of words that a model can process.
    [Page 5] Chapter 5: Word Frequency Analysis Word frequency analysis counts how many times each word appears in a document.
    [Page 5] 

In [ ]:
# =====================================================================
# TASK 2: Structured Page Metadata Extraction and CSV Export
# =====================================================================
print("\n" + "=" * 60, "\nTASK 2\n" + "=" * 60)

# (a) build pages_data
pages_data = []
for i, text in enumerate(read_pdf_pages(PDF_FILE), start=1):
    pages_data.append({
        "page_number": i,
        "raw_text": text,
        "word_count": len(text.split()),
        "char_count": len(re.sub(r"\s", "", text)),   # characters excluding spaces
    })
print("(a) First 2 entries:")
for entry in pages_data[:2]:
    print("   ", entry)

# (b) highest / lowest non-zero / average
highest = max(pages_data, key=lambda d: d["word_count"])
print(f"(b-i) Highest word count: page {highest['page_number']} ({highest['word_count']} words)")
print("      Content:\n", highest["raw_text"])

non_zero = [d for d in pages_data if d["word_count"] > 0]
if non_zero:
    lowest = min(non_zero, key=lambda d: d["word_count"])
    print(f"(b-ii) Lowest non-zero word count: page {lowest['page_number']} ({lowest['word_count']} words)")
else:
    print("(b-ii) All pages have zero words.")

if pages_data:
    average = round(sum(d["word_count"] for d in pages_data) / len(pages_data), 2)
    print("(b-iii) Average word count:", average)


# (c) phrase search with custom exception
class PhraseNotFoundError(Exception):
    """Raised when the phrase does not appear on any page."""
    pass


phrase = input("(c) Enter a search phrase: ").strip()
try:
    matches = []
    for d in pages_data:
        n = d["raw_text"].lower().count(phrase.lower()) if phrase else 0
        if n > 0:
            matches.append((d["page_number"], n))
    if not matches:
        raise PhraseNotFoundError(f"The phrase '{phrase}' was not found on any page.")
    for page_no, n in matches:
        print(f"    Page {page_no}: appears {n} time(s)")
except PhraseNotFoundError as e:
    print("    Error caught:", e)

# (d) write CSV using csv module then read it back
with open("pdf_summary.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["page_number", "word_count", "char_count"])
    for d in pages_data:
        writer.writerow([d["page_number"], d["word_count"], d["char_count"]])

print("(d) Contents of pdf_summary.csv:")
with open("pdf_summary.csv", "r", newline="", encoding="utf-8") as f:
    for row in csv.reader(f):
        print("   ", row)




TASK 2
(a) First 2 entries:
    {'page_number': 1, 'raw_text': 'Chapter 1: Introduction to Natural Language\nProcessing\nNatural language processing, often called NLP, is a field of artificial intelligence that helps\ncomputers understand, interpret and generate human language. It combines linguistics, computer\nscience and machine learning.\nEarly NLP systems in the 1950s relied on hand written rules. Modern systems learn patterns from\nvery large collections of text called corpora. Today, NLP powers search engines, translation tools,\nchatbots and voice assistants.\nIn 2023 alone, researchers published more than 5000 papers on language models. You can read\nmore at https://www.aclweb.org and https://arxiv.org for free.', 'word_count': 92, 'char_count': 566}
    {'page_number': 2, 'raw_text': "Chapter 2: Text Tokenization\nTokenization is the process of splitting text into smaller units called tokens. A token can be a word,\na number or a punctuation mark. Good tokenization is the fi

In [ ]:

# =====================================================================
# TASK 3: PDF Text Preprocessing Pipeline for NLP
# =====================================================================
print("\n" + "=" * 60, "\nTASK 3\n" + "=" * 60)
import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

# (a) extract all text and clean in the required order
text = "\n".join(read_pdf_pages(PDF_FILE))
before_chars = len(text)

text = re.sub(r"https?://\S+|www\.\S+", "", text)   # (i) remove URLs
text = re.sub(r"\d", "", text)                        # (ii) remove digits
# (iii) keep only letters, spaces and full stops. Replacing with a SPACE (not "")
#       stops words on different lines from getting glued together.
text = re.sub(r"[^A-Za-z. ]", " ", text)
text = re.sub(r" +", " ", text)                       # (iv) collapse multiple spaces
print(f"(a) Characters before cleaning: {before_chars}")
print(f"    Characters after cleaning : {len(text)}")

# (b) lowercase, split into sentences, strip, remove empties
text = text.lower()
raw_sentences = re.split(r"[.!?]", text)
print("(b) Sentence count BEFORE removing empty strings:", len(raw_sentences))
sentences = [s.strip() for s in raw_sentences]
sentences = [s for s in sentences if s]
print("    Sentence count AFTER removing empty strings :", len(sentences))

# (c) tokenize, remove stopwords, drop sentences with < 5 remaining tokens
stop_en = set(stopwords.words("english"))
print("(c) Sentence count BEFORE the filter:", len(sentences))
kept = []
for s in sentences:
    words = re.findall(r"[a-z]+", s)
    meaningful = [w for w in words if w not in stop_en]
    if len(meaningful) >= 5:
        kept.append(s)
print("    Sentence count AFTER the filter :", len(kept))
print("    3 sample sentences:")
for s in kept[:3]:
    print("     -", s)

# (d) write to file, re-read, report
with open("nlp_ready_output.txt", "w", encoding="utf-8") as f:
    for s in kept:
        f.write(s + "\n")

with open("nlp_ready_output.txt", "r", encoding="utf-8") as f:
    lines = [line.rstrip("\n") for line in f]
print("(d) (i) Total lines written:", len(lines))
if lines:
    print("    (ii) Longest sentence :", max(lines, key=len))
    print("    (iii) Shortest sentence:", min(lines, key=len))




TASK 3
(a) Characters before cleaning: 3720
    Characters after cleaning : 3598
(b) Sentence count BEFORE removing empty strings: 49
    Sentence count AFTER removing empty strings : 48
(c) Sentence count BEFORE the filter: 48
    Sentence count AFTER the filter : 43
    3 sample sentences:
     - chapter introduction to natural language processing natural language processing often called nlp is a field of artificial intelligence that helps computers understand interpret and generate human language
     - it combines linguistics computer science and machine learning
     - early nlp systems in the s relied on hand written rules
(d) (i) Total lines written: 43
    (ii) Longest sentence : chapter introduction to natural language processing natural language processing often called nlp is a field of artificial intelligence that helps computers understand interpret and generate human language
    (iii) Shortest sentence: simple tokenizers split text on whitespace


In [ ]:

# =====================================================================
# TASK 4: DOCX Paragraph Classification and Statistical Analysis
# =====================================================================
print("\n" + "=" * 60, "\nTASK 4\n" + "=" * 60)

# (a) store text and style name of every paragraph
doc = Document(DOCX_FILE)
para_list = [{"text": p.text, "style": p.style.name} for p in doc.paragraphs]
print("(a) Total paragraphs:", len(para_list))
print("    Unique style names:", sorted({p["style"] for p in para_list}))

# (b) classify into exactly three groups
headings, body, other = [], [], []
for p in para_list:
    if not p["text"].strip():
        other.append(p)                                  # blank -> Other/Empty
    elif p["style"].startswith("Heading"):
        headings.append(p)
    elif p["style"] == "Normal" or "Body" in p["style"]:
        body.append(p)
    else:
        other.append(p)
print(f"(b) Headings: {len(headings)} | Body: {len(body)} | Other/Empty: {len(other)}")

# (c) statistics for body paragraphs with more than 5 words
qualifying = []
for p in body:
    wc = len(p["text"].split())
    if wc > 5:
        qualifying.append({
            "text": p["text"],
            "word_count": wc,
            "char_count": len(p["text"].replace(" ", "")),
        })
if qualifying:
    hi = max(qualifying, key=lambda d: d["word_count"])
    lo = min(qualifying, key=lambda d: d["word_count"])
    mean_wc = round(sum(d["word_count"] for d in qualifying) / len(qualifying), 2)
    print(f"(c) (i) Highest ({hi['word_count']} words, {hi['char_count']} chars): {hi['text']}")
    print(f"    (ii) Lowest  ({lo['word_count']} words, {lo['char_count']} chars): {lo['text']}")
    print("    (iii) Mean word count:", mean_wc)
else:
    print("(c) No body paragraph has more than 5 words.")

# (d) nested dictionary by style, sorted by count (descending)
style_dict = {}
for p in para_list:
    entry = style_dict.setdefault(p["style"], {"count": 0, "texts": []})
    entry["count"] += 1
    entry["texts"].append(p["text"])
style_dict = dict(sorted(style_dict.items(), key=lambda kv: kv[1]["count"], reverse=True))
for style, info in style_dict.items():
    if info["count"] > 2:
        info["texts"] = info["texts"][:2]                # show only first 2 texts
print("(d) Style dictionary:")
pprint.pprint(style_dict, width=100)



TASK 4
(a) Total paragraphs: 14
    Unique style names: ['Heading 1', 'Normal', 'Title']
(b) Headings: 3 | Body: 9 | Other/Empty: 2
(c) (i) Highest (18 words, 95 chars): Overall the class performed well and most students showed a good understanding of the basics of text processing.
    (ii) Lowest  (9 words, 36 chars): Lab work is graded out of one hundred marks.
    (iii) Mean word count: 14.0
(d) Style dictionary:
{'Heading 1': {'count': 3, 'texts': ['Course Overview', 'Student Results']},
 'Normal': {'count': 10,
            'texts': ['This report summarises the performance of students in the NLP course during '
                      'the fall semester.',
                      '']},
 'Title': {'count': 1, 'texts': ['Student Course Report']}}


In [ ]:

# =====================================================================
# TASK 5: DOCX Table Extraction and Pandas Integration
# =====================================================================
print("\n" + "=" * 60, "\nTASK 5\n" + "=" * 60)

# (a) extract every table as a list of lists
doc = Document(DOCX_FILE)
all_tables = []
for table in doc.tables:
    all_tables.append([[cell.text.strip() for cell in row.cells] for row in table.rows])
print("(a) Total tables found:", len(all_tables))
for i, t in enumerate(all_tables, start=1):
    cols = len(t[0]) if t else 0
    print(f"    Table {i}: {len(t)} rows x {cols} columns")

# (b) first table -> DataFrame, count empty cells per column
if not all_tables:
    print("    No tables in this document. Use a .docx that contains a table.")
    sys.exit()
first = all_tables[0]
df = pd.DataFrame(first[1:], columns=first[0])
print("(b) DataFrame:")
print(df)
empty_counts = df.apply(lambda col: col.map(lambda v: v is None or str(v).strip() == "").sum())
print("    Empty/None cells per column:")
print(empty_counts)

# (c) paragraphs before the first table
pre_table_paragraphs = []
for child in doc.element.body.iterchildren():
    if child.tag == qn("w:tbl"):
        break                                            # stop at the first table
    if child.tag == qn("w:p"):
        pre_table_paragraphs.append(Paragraph(child, doc).text)
print("(c) Paragraphs before first table:", len(pre_table_paragraphs))

# (d) write labeled sections to a file
with open("docx_extracted.txt", "w", encoding="utf-8") as f:
    f.write("--- PRE-TABLE PARAGRAPHS ---\n")
    for para in pre_table_paragraphs:
        f.write(para + "\n")
    f.write("\n--- TABLE DATA ---\n")
    for row in first:
        f.write(" | ".join(row) + "\n")
print("(d) Written docx_extracted.txt")
with open("docx_extracted.txt", encoding="utf-8") as f:
    print(f.read())



TASK 5
(a) Total tables found: 1
    Table 1: 5 rows x 4 columns
(b) DataFrame:
         Name Marks Grade           Remarks
0    Ali Khan    88     A    Excellent work
1  Sara Ahmed    76     B                  
2  Usman Raza           C   Absent in lab 2
3  Hina Malik    92     A  Top of the class
    Empty/None cells per column:
Name       0
Marks      1
Grade      0
Remarks    1
dtype: int64
(c) Paragraphs before first table: 11
(d) Written docx_extracted.txt
--- PRE-TABLE PARAGRAPHS ---
Student Course Report
This report summarises the performance of students in the NLP course during the fall semester.

Short note.
All students attended the labs regularly and submitted their assignments before the deadline.
Course Overview
The course covers regular expressions, tokenization, text cleaning, word frequency analysis and working with APIs.
Students learned how to extract text from PDF and Word documents using Python libraries.
Lab work is graded out of one hundred marks.
Student Result

In [ ]:

# =====================================================================
# TASK 6: Multi-Document Comparison, Deduplication, Frequency Report
# =====================================================================
print("\n" + "=" * 60, "\nTASK 6\n" + "=" * 60)

doc_a, doc_b = Document(DOCX_A), Document(DOCX_B)

# (a) headings + set operations
heads_a = [p.text.strip() for p in doc_a.paragraphs
           if p.style.name.startswith("Heading") and p.text.strip()]
heads_b = [p.text.strip() for p in doc_b.paragraphs
           if p.style.name.startswith("Heading") and p.text.strip()]
set_a, set_b = set(heads_a), set(heads_b)
print("(a) (i) Common headings      :", sorted(set_a & set_b))
print("    (ii) Only in Document A  :", sorted(set_a - set_b))
print("    (iii) Only in Document B :", sorted(set_b - set_a))

# (b) Normal paragraphs, merge, deduplicate (stripped + lowercased comparison)
normal_a = [p.text for p in doc_a.paragraphs if p.style.name == "Normal" and p.text.strip()]
normal_b = [p.text for p in doc_b.paragraphs if p.style.name == "Normal" and p.text.strip()]
merged = normal_a + normal_b
seen, unique_paras = set(), []
for para in merged:
    key = para.strip().lower()
    if key not in seen:
        seen.add(key)
        unique_paras.append(para)
print(f"(b) Paragraphs before dedup: {len(merged)} | after dedup: {len(unique_paras)}")

# (c) tokenise, Counter, remove stopwords, top 20
all_tokens = []
for para in unique_paras:
    all_tokens.extend(re.sub(r"[^a-z\s]", "", para.lower()).split())
counter = Counter(w for w in all_tokens if w not in STOPWORDS)
print("(c) Top 20 words:")
for word, count in counter.most_common(20):
    print(f"    {word:<15} {count}")

# (d) summary
lengths = [len(p.split()) for p in unique_paras]
print("(d) SUMMARY")
print("    (i) Total headings (both docs)     :", len(heads_a) + len(heads_b))
print("    (ii) Unique body paragraphs        :", len(unique_paras))
print("    (iii) Vocabulary size (word types) :", len(set(all_tokens)))
if lengths:
    print("    (iv) Average paragraph length      :", round(sum(lengths) / len(lengths), 2), "words")
    longest = max(unique_paras, key=lambda p: len(p.split()))
    print(f"    (v) Longest paragraph ({len(longest.split())} words):\n        {longest}")




TASK 6
(a) (i) Common headings      : ['Introduction', 'Tokenization']
    (ii) Only in Document A  : ['Regular Expressions']
    (iii) Only in Document B : ['Word Frequency']
(b) Paragraphs before dedup: 10 | after dedup: 9
(c) Top 20 words:
    language        4
    text            4
    words           3
    natural         2
    ways            2
    before          2
    numbers         2
    different       2
    processing      1
    allows          1
    computers       1
    work            1
    human           1
    useful          1
    preprocessing   1
    essential       1
    step            1
    any             1
    analysis        1
    data            1
(d) SUMMARY
    (i) Total headings (both docs)     : 6
    (ii) Unique body paragraphs        : 9
    (iii) Vocabulary size (word types) : 102
    (iv) Average paragraph length      : 14.78 words
    (v) Longest paragraph (17 words):
        Patterns should always be tested on small samples of text before they are 

In [ ]:

# =====================================================================
# Helper for API tasks (7, 8, 9)
# =====================================================================
def fetch_json(url):
    """GET a URL; raise ConnectionError if the status code is not exactly 200."""
    response = requests.get(url, timeout=15)
    if response.status_code != 200:
        raise ConnectionError(f"Request to {url} failed with status code {response.status_code}")
    return response.json()


# =====================================================================
# TASK 7: Multi-Endpoint API Data Joining and Analysis
# =====================================================================
print("\n" + "=" * 60, "\nTASK 7\n" + "=" * 60)

# (a) fetch both endpoints
users = fetch_json("https://jsonplaceholder.typicode.com/users")
posts = fetch_json("https://jsonplaceholder.typicode.com/posts")
print(f"(a) Fetched {len(users)} users and {len(posts)} posts")

# (b) join on users.id == posts.userId
joined = {}
for u in users:
    user_posts = [p["title"] for p in posts if p["userId"] == u["id"]]
    joined[u["name"]] = {"email": u["email"], "post_count": len(user_posts), "titles": user_posts}
print("(b) First 3 users:")
for name in list(joined)[:3]:
    print(f"  {name}: ")
    pprint.pprint(joined[name], width=100)

# (c) user with the most posts
top_name = max(joined, key=lambda n: joined[n]["post_count"])
top_user = next(u for u in users if u["name"] == top_name)
print("(c) Most posts:")
print("    (i) Name   :", top_name)
print("    (ii) Email :", joined[top_name]["email"])
print("    (iii) Company:", top_user["company"]["name"])
print("    (iv) Titles:")
for i, title in enumerate(joined[top_name]["titles"], start=1):
    print(f"        {i}. {title}")

# (d) filter: company.bs has "synergies" OR city longer than 8 characters
print("(d) Matching users:")
for u in users:
    if "synergies" in u["company"]["bs"].lower() or len(u["address"]["city"]) > 8:
        print(f"    {u['name']} | {u['address']['city']} | posts: {joined[u['name']]['post_count']}")



TASK 7
(a) Fetched 10 users and 100 posts
(b) First 3 users:
  Leanne Graham: 
{'email': 'Sincere@april.biz',
 'post_count': 10,
 'titles': ['sunt aut facere repellat provident occaecati excepturi optio reprehenderit',
            'qui est esse',
            'ea molestias quasi exercitationem repellat qui ipsa sit aut',
            'eum et est occaecati',
            'nesciunt quas odio',
            'dolorem eum magni eos aperiam quia',
            'magnam facilis autem',
            'dolorem dolore est ipsam',
            'nesciunt iure omnis dolorem tempora et accusantium',
            'optio molestias id quia eum']}
  Ervin Howell: 
{'email': 'Shanna@melissa.tv',
 'post_count': 10,
 'titles': ['et ea vero quia laudantium autem',
            'in quibusdam tempore odit est dolorem',
            'dolorum ut in voluptas mollitia et saepe quo animi',
            'voluptatem eligendi optio',
            'eveniet quod temporibus',
            'sint suscipit perspiciatis velit dolorum rer

In [ ]:

# =====================================================================
# TASK 8: To-Do Completion Rate Analysis Across Users
# =====================================================================
print("\n" + "=" * 60, "\nTASK 8\n" + "=" * 60)

# (a) fetch + validate
resp = requests.get("https://jsonplaceholder.typicode.com/todos", timeout=15)
if resp.status_code != 200:
    print(f"Error: status code was {resp.status_code}, expected 200.")
    sys.exit()
todos = resp.json()
if not isinstance(todos, list):
    print("Error: response is not a list.")
    sys.exit()
if len(todos) < 1:
    print("Error: the list has no records.")
    sys.exit()
print("(a) Total todo records:", len(todos))

# (b) group by userId
stats = {}
for t in todos:
    s = stats.setdefault(t["userId"], {"total": 0, "completed": 0, "incomplete": 0})
    s["total"] += 1
    if t["completed"]:
        s["completed"] += 1
    else:
        s["incomplete"] += 1
print("(b) Per-user stats:")
for uid in sorted(stats):
    s = stats[uid]
    print(f"    User {uid}: {s}  -> completed + incomplete == total: "
          f"{s['completed'] + s['incomplete'] == s['total']}")

# (c) completion rates
rates = {uid: round(s["completed"] / s["total"] * 100, 1) for uid, s in stats.items()}
best = max(rates, key=rates.get)
worst = min(rates, key=rates.get)
print(f"(c) (i) Highest rate: user {best} ({rates[best]}%)")
print(f"    (ii) Lowest rate : user {worst} ({rates[worst]}%)")
print(f"    (iii) Overall average: {round(sum(rates.values()) / len(rates), 1)}%")

# (d) lookup + formatted table
users8 = fetch_json("https://jsonplaceholder.typicode.com/users")
lookup = {u["id"]: u["name"] for u in users8}
ranked = sorted(rates.items(), key=lambda kv: kv[1], reverse=True)
print("(d)")
print(f"{'Rank'.ljust(6)}{'Name'.ljust(28)}{'Total Tasks'.ljust(14)}{'Completed'.ljust(12)}Completion Rate %")
print("-" * 76)
for rank, (uid, rate) in enumerate(ranked, start=1):
    s = stats[uid]
    print(f"{str(rank).ljust(6)}{lookup.get(uid, 'Unknown').ljust(28)}"
          f"{str(s['total']).ljust(14)}{str(s['completed']).ljust(12)}{rate}")



TASK 8
(a) Total todo records: 200
(b) Per-user stats:
    User 1: {'total': 20, 'completed': 11, 'incomplete': 9}  -> completed + incomplete == total: True
    User 2: {'total': 20, 'completed': 8, 'incomplete': 12}  -> completed + incomplete == total: True
    User 3: {'total': 20, 'completed': 7, 'incomplete': 13}  -> completed + incomplete == total: True
    User 4: {'total': 20, 'completed': 6, 'incomplete': 14}  -> completed + incomplete == total: True
    User 5: {'total': 20, 'completed': 12, 'incomplete': 8}  -> completed + incomplete == total: True
    User 6: {'total': 20, 'completed': 6, 'incomplete': 14}  -> completed + incomplete == total: True
    User 7: {'total': 20, 'completed': 9, 'incomplete': 11}  -> completed + incomplete == total: True
    User 8: {'total': 20, 'completed': 11, 'incomplete': 9}  -> completed + incomplete == total: True
    User 9: {'total': 20, 'completed': 8, 'incomplete': 12}  -> completed + incomplete == total: True
    User 10: {'total': 20,

In [ ]:

# =====================================================================
# TASK 9: Nested JSON Flattening, DataFrame Operations, Comment Analysis
# =====================================================================
print("\n" + "=" * 60, "\nTASK 9\n" + "=" * 60)


# (a) flatten function
def flatten_user(user: dict) -> dict:
    return {
        "id": user["id"],
        "name": user["name"],
        "username": user["username"],
        "email": user["email"],
        "city": user["address"]["city"],
        "zipcode": user["address"]["zipcode"],
        "lat": user["address"]["geo"]["lat"],
        "lng": user["address"]["geo"]["lng"],
        "company_name": user["company"]["name"],
        "company_catchphrase": user["company"]["catchPhrase"],
    }


users9 = fetch_json("https://jsonplaceholder.typicode.com/users")
flat_users = [flatten_user(u) for u in users9]
print("(a) Flattened users:")
pprint.pprint(flat_users, width=110)

# (b) DataFrame operations
df9 = pd.DataFrame(flat_users)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
print("(b) DataFrame:")
print(df9)
df9["lat"] = df9["lat"].astype(float)                                    # (i)
df9["lng"] = df9["lng"].astype(float)
df9["distance_from_origin"] = df9.apply(                                 # (ii)
    lambda row: math.sqrt(row["lat"] ** 2 + row["lng"] ** 2), axis=1)
farthest = df9.loc[df9["distance_from_origin"].idxmax()]                 # (iii)
print(f"    Greatest distance from origin: {farthest['name']} ({farthest['distance_from_origin']:.2f})")

# (c) hemispheres
north = df9[df9["lat"] > 0]
south = df9[df9["lat"] < 0]
print("(c) Northern Hemisphere:", list(north["name"]))
print("    Southern Hemisphere:", list(south["name"]))
avg_n = north["distance_from_origin"].mean() if len(north) else 0
avg_s = south["distance_from_origin"].mean() if len(south) else 0
print(f"    Avg distance North: {avg_n:.2f} | South: {avg_s:.2f}")
print("    Higher average distance:", "Northern" if avg_n > avg_s else "Southern", "Hemisphere")

# (d) comment count per post
posts9 = fetch_json("https://jsonplaceholder.typicode.com/posts")
comments9 = fetch_json("https://jsonplaceholder.typicode.com/comments")
comment_counts = Counter(c["postId"] for c in comments9)
result = [(p["id"], p["title"][:40], comment_counts.get(p["id"], 0)) for p in posts9]
result.sort(key=lambda t: t[2], reverse=True)
print("(d) Top 5 most-commented posts:")
for t in result[:5]:
    print("   ", t)
print("    Bottom 5 least-commented posts:")
for t in result[-5:]:
    print("   ", t)



TASK 9
(a) Flattened users:
[{'city': 'Gwenborough',
  'company_catchphrase': 'Multi-layered client-server neural-net',
  'company_name': 'Romaguera-Crona',
  'email': 'Sincere@april.biz',
  'id': 1,
  'lat': '-37.3159',
  'lng': '81.1496',
  'name': 'Leanne Graham',
  'username': 'Bret',
  'zipcode': '92998-3874'},
 {'city': 'Wisokyburgh',
  'company_catchphrase': 'Proactive didactic contingency',
  'company_name': 'Deckow-Crist',
  'email': 'Shanna@melissa.tv',
  'id': 2,
  'lat': '-43.9509',
  'lng': '-34.4618',
  'name': 'Ervin Howell',
  'username': 'Antonette',
  'zipcode': '90566-7771'},
 {'city': 'McKenziehaven',
  'company_catchphrase': 'Face to face bifurcated interface',
  'company_name': 'Romaguera-Jacobson',
  'email': 'Nathan@yesenia.net',
  'id': 3,
  'lat': '-68.6102',
  'lng': '-47.0653',
  'name': 'Clementine Bauch',
  'username': 'Samantha',
  'zipcode': '59590-4157'},
 {'city': 'South Elvis',
  'company_catchphrase': 'Multi-tiered zero tolerance productivity',
  'c